In [2]:
import pandas as pd
import optuna

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

ModuleNotFoundError: No module named 'pandas'

In [33]:
train_df = pd.read_csv('../data/processed/train.csv')

X_train = train_df.drop(columns='Survived')
y_train = train_df['Survived']

test_df = pd.read_csv('../data/processed/test.csv')

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Hyperparams for RF

In [34]:
def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestClassifier(**params)

    score = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1
    ).mean()

    return score

In [35]:
rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective, n_trials=50)

print('Best RF score:', rf_study.best_value)
print('Best RF params:', rf_study.best_params)

[I 2026-05-15 18:44:31,742] A new study created in memory with name: no-name-b2ddb834-a56c-41c7-8f36-0d0c4ffd116b
[I 2026-05-15 18:44:32,868] Trial 0 finished with value: 0.8226602222082733 and parameters: {'n_estimators': 471, 'max_depth': 10, 'min_samples_split': 16, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8226602222082733.
[I 2026-05-15 18:44:33,906] Trial 1 finished with value: 0.8192894356914191 and parameters: {'n_estimators': 439, 'max_depth': 13, 'min_samples_split': 9, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 0 with value: 0.8226602222082733.
[I 2026-05-15 18:44:34,252] Trial 2 finished with value: 0.8282593685267716 and parameters: {'n_estimators': 128, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.8282593685267716.
[I 2026-05-15 18:44:35,138] Trial 3 finished with value: 0.8181658401858012 and parameters: {'n_estimators': 321, 'max_depth': 10, '

Best RF score: 0.8361245370660976
Best RF params: {'n_estimators': 137, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': None}


# Hyperparams for CatBoost

In [36]:
def catboost_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 600),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_seed': 42,
        'verbose': 0,
        'loss_function': 'Logloss',
        'eval_metric': 'Accuracy'
    }

    model = CatBoostClassifier(**params)

    score = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1
    ).mean()

    return score

In [37]:
cat_study = optuna.create_study(direction='maximize')
cat_study.optimize(catboost_objective, n_trials=50)

print('Best Catboost score:', cat_study.best_value)
print('Best Catboost params:', cat_study.best_params)

[I 2026-05-15 18:45:45,203] A new study created in memory with name: no-name-a745836a-afb6-4aa8-81a7-10059d073352
[I 2026-05-15 18:45:46,728] Trial 0 finished with value: 0.8327537505492437 and parameters: {'iterations': 460, 'depth': 3, 'learning_rate': 0.11801055096571757, 'l2_leaf_reg': 3.6759462283652242}. Best is trial 0 with value: 0.8327537505492437.
[I 2026-05-15 18:45:48,016] Trial 1 finished with value: 0.8293829640323898 and parameters: {'iterations': 400, 'depth': 3, 'learning_rate': 0.13406509808028935, 'l2_leaf_reg': 6.02174092681887}. Best is trial 0 with value: 0.8327537505492437.
[I 2026-05-15 18:45:49,309] Trial 2 finished with value: 0.8361057058565061 and parameters: {'iterations': 318, 'depth': 4, 'learning_rate': 0.03847720710024565, 'l2_leaf_reg': 3.508837192769716}. Best is trial 2 with value: 0.8361057058565061.
[I 2026-05-15 18:45:50,517] Trial 3 finished with value: 0.8361119829263698 and parameters: {'iterations': 264, 'depth': 5, 'learning_rate': 0.03547557

Best Catboost score: 0.8406000878789781
Best Catboost params: {'iterations': 421, 'depth': 8, 'learning_rate': 0.03555080953960718, 'l2_leaf_reg': 9.911518120395717}


Catboost gave little bit better result. Let's try it on test dataset.

In [40]:
print(test_df.columns)
print(train_df.columns)

Index(['Unnamed: 0', 'Pclass', 'Age', 'Fare', 'Age_bin', 'Fare_bin',
       'Ticket_group', 'Initial', 'Family_size', 'Is_alone', 'Embarked_Q',
       'Embarked_S', 'Sex_male', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E',
       'Deck_F', 'Deck_G', 'Deck_T', 'Deck_Unknown'],
      dtype='object')
Index(['Survived', 'Unnamed: 0', 'Pclass', 'Age', 'Fare', 'Age_bin',
       'Fare_bin', 'Ticket_group', 'Initial', 'Family_size', 'Is_alone',
       'Embarked_Q', 'Embarked_S', 'Sex_male', 'Deck_B', 'Deck_C', 'Deck_D',
       'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Deck_Unknown'],
      dtype='object')


In [41]:
model = CatBoostClassifier(
    **cat_study.best_params,
    random_seed=42,
    verbose=0,
    loss_function='Logloss',
    eval_metric='Accuracy'
)


X_test = test_df.drop(columns=["PassengerId"], errors="ignore")
X_test = X_test.reindex(columns=X.columns, fill_value=0)

model.fit(X, y)

raw_test = pd.read_csv('../data/raw/test.csv')
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId':raw_test['PassengerId'],
    'Survived': predictions.astype(int)
})

submission.to_csv('../submission.csv', index=False)